In [40]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph.message import add_messages
import sqlite3

llm = ChatAnthropic(
    model_name='claude-opus-4-7',
        base_url='https://api.agentshop247.com/api/claude',
        temperature=0,
        timeout=None,
        stop=None,
)

def message_limit(old: list[BaseMessage], new: list[BaseMessage]):
    merge = old + new
    return merge[-5:]

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

def receive_input(state: AgentState):
    prompt_input = input("User: ")
    return {
        'messages': [HumanMessage(content = prompt_input)]
    }
def process(state: AgentState) -> dict:
    full_response = llm.invoke(state['messages'])
    for block in full_response.content:
        if isinstance(block, dict) and block['type'] == 'text':
            response = block['text']
            print(f"Agent: {response}")
            break
    return {
        'messages': [full_response]
    }

def router(state: AgentState):
    if state['messages'][-1].content == 'exit':
        return "end"
    else: 
        return "llm"

graph = StateGraph(AgentState)
graph.add_node('agent', process)
graph.add_node('human', receive_input)
graph.add_edge(START, 'human')
graph.add_edge('agent', END)
graph.add_edge('agent', 'human')
graph.add_conditional_edges(
    'human', 
    router,
    {
        'end': END,
        'llm': 'agent'
    }
)


conn = sqlite3.connect("test.sqlite", check_same_thread = False)
memory = SqliteSaver(conn)

app = graph.compile(checkpointer=memory)

config = {"configurable":{"thread_id":"1"}}

res = app.invoke({"messages": []},config=config)


Agent: Nice to meet you, Nam! I'm Claude. How can I help you today?
Agent: Got it, Nam! Thanks for sharing. At 20, you're at an exciting stage — whether you're finishing up university, starting a career, or figuring things out. Is there anything I can help you with today?
Agent: That's great, Nam! Machine Learning with LangGraph is a powerful combination — LangGraph is particularly useful for building AI agents and complex LLM-based workflows with state management.

Are you working on a particular project, or is there something specific you'd like help with? Happy to assist with:

- Understanding LangGraph concepts (nodes, edges, state machines)
- Building agentic workflows
- ML fundamentals or model implementation
- Debugging code
- Explaining a concept

What are you diving into?


In [41]:
history = list(app.get_state_history(config = config))
for block in history:    # Block is StateSnapshot
    message = block.values['messages']
    for i in message:
        if isinstance(i, HumanMessage):
            print("Human: ", i.content, end = " ")
            print("|",end = " ")
        elif isinstance(i, AIMessage):
            print("AI: ", end = " ")
            for j in i.content:
                if isinstance(j, dict) and j['type'] == 'text':
                    print(j['text'], end = " ")
                    break
            print("|", end = " ")
    print(block.config['configurable']['checkpoint_id'], block.next)

Human:  I'm Nam | AI:  Nice to meet you, Nam! I'm Claude. How can I help you today? | Human:  20 years old | AI:  Got it, Nam! Thanks for sharing. At 20, you're at an exciting stage — whether you're finishing up university, starting a career, or figuring things out. Is there anything I can help you with today? | Human:  studying MachineLearning, LangGraph | AI:  That's great, Nam! Machine Learning with LangGraph is a powerful combination — LangGraph is particularly useful for building AI agents and complex LLM-based workflows with state management.

Are you working on a particular project, or is there something specific you'd like help with? Happy to assist with:

- Understanding LangGraph concepts (nodes, edges, state machines)
- Building agentic workflows
- ML fundamentals or model implementation
- Debugging code
- Explaining a concept

What are you diving into? | Human:  exit | 1f19bd7a-3268-6e32-8007-39b5552f09f6 ()
Human:  I'm Nam | AI:  Nice to meet you, Nam! I'm Claude. How can 

In [42]:
config_rewind = {
                "configurable": {
                    "thread_id": "1",
                    "checkpoint_id": "1f19bd74-ea6c-64e4-8004-604b5077497e"
                    }
}   
app.invoke(None,config= config_rewind)

Agent: That's a cool combination — economics and content creation. There's actually a lot of overlap there these days. Understanding markets, consumer behavior, and incentives from economics can really inform how you create and position content.

Are you doing content creation on the side while studying, or more focused on it? And what kind of content are you into — videos, social media, writing?


{'messages': [HumanMessage(content="I'm Nam", additional_kwargs={}, response_metadata={}, id='a940130c-513f-4946-9a95-49f7831953e2'),
  AIMessage(content=[{'signature': 'f0c22c89e9af69deee6b22b54ccab4581f687ca5d18d0d4c2c7d70ac761fcd4e', 'thinking': "The user is introducing themselves as Nam. This is a simple greeting/introduction. I should respond in a friendly, welcoming way and invite them to share what they'd like to talk about or how I can help.", 'type': 'thinking'}, {'text': "Nice to meet you, Nam! I'm Claude. How can I help you today?", 'type': 'text'}], additional_kwargs={}, response_metadata={'id': '06d4eb718cd8bf804a923157dd0125e8', 'container': None, 'model': 'claude-opus-4-7', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': None, 'cache_creation_input_tokens': None, 'cache_read_input_tokens': None, 'inference_geo': None, 'input_tokens': 140, 'output_tokens': 63, 'output_tokens_details': None, 'server_tool_use': None, 'serv

In [43]:
history = list(app.get_state_history(config = config))
for block in history:    # Block is StateSnapshot
    message = block.values['messages']
    for i in message:
        if isinstance(i, HumanMessage):
            print("Human: ", i.content, end = " ")
            print("|",end = " ")
        elif isinstance(i, AIMessage):
            print("AI: ", end = " ")
            for j in i.content:
                if isinstance(j, dict) and j['type'] == 'text':
                    print(j['text'], end = " ")
                    break
            print("|", end = " ")
    print(block.config['configurable']['checkpoint_id'], block.next)

Human:  I'm Nam | AI:  Nice to meet you, Nam! I'm Claude. How can I help you today? | Human:  20 years old | AI:  Got it, Nam! Thanks for sharing. At 20, you're at an exciting stage — whether you're finishing up university, starting a career, or figuring things out. Is there anything I can help you with today? | Human:  studying economics, content creators | AI:  That's a cool combination — economics and content creation. There's actually a lot of overlap there these days. Understanding markets, consumer behavior, and incentives from economics can really inform how you create and position content.

Are you doing content creation on the side while studying, or more focused on it? And what kind of content are you into — videos, social media, writing? | Human:  exit | 1f19bd82-7a41-6bfc-8008-36cc1441ae8b ()
Human:  I'm Nam | AI:  Nice to meet you, Nam! I'm Claude. How can I help you today? | Human:  20 years old | AI:  Got it, Nam! Thanks for sharing. At 20, you're at an exciting stage — 

Đã quên Machine Learning và LangGraph

In [45]:
res = app.invoke({'messages': []},config=config)

Agent: Bạn đã nói với tôi:

1. **Tên**: Nam
2. **Tuổi**: 20
3. **Học**: Kinh tế (Economics)
4. **Làm**: Content creator

Đúng không? 😊


Để quay lại Machine Learning và LangGraph

In [51]:
new_config = {"configurable":{"thread_id": "1","checkpoint_id":"1f19bd7a-3268-6e32-8007-39b5552f09f6"}}

app.invoke({'messages': []}, config = new_config)

Agent: ## Tóm tắt đoạn hội thoại

**Người dùng:**
- **Tên:** Nam
- **Tuổi:** 20
- **Lĩnh vực đang học:** Machine Learning và LangGraph

**Nội dung:**
Đây là buổi làm quen, Nam giới thiệu về bản thân và lĩnh vực đang nghiên cứu. LangGraph là thư viện dùng để xây dựng các agent AI và workflow phức tạp dựa trên LLM với quản lý trạng thái.

---

Bạn cần hỗ trợ thêm gì không? 😊


{'messages': [HumanMessage(content="I'm Nam", additional_kwargs={}, response_metadata={}, id='a940130c-513f-4946-9a95-49f7831953e2'),
  AIMessage(content=[{'signature': 'f0c22c89e9af69deee6b22b54ccab4581f687ca5d18d0d4c2c7d70ac761fcd4e', 'thinking': "The user is introducing themselves as Nam. This is a simple greeting/introduction. I should respond in a friendly, welcoming way and invite them to share what they'd like to talk about or how I can help.", 'type': 'thinking'}, {'text': "Nice to meet you, Nam! I'm Claude. How can I help you today?", 'type': 'text'}], additional_kwargs={}, response_metadata={'id': '06d4eb718cd8bf804a923157dd0125e8', 'container': None, 'model': 'claude-opus-4-7', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': None, 'cache_creation_input_tokens': None, 'cache_read_input_tokens': None, 'inference_geo': None, 'input_tokens': 140, 'output_tokens': 63, 'output_tokens_details': None, 'server_tool_use': None, 'serv